# Model as an API: FastAPI and Flask

Goal: show the simplest way to move a model from a notebook into an application.


In [ ]:
from pathlib import Path
import joblib
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

Path("models").mkdir(exist_ok=True)
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
acc = accuracy_score(y_test, model.predict(X_test))

artifact = {
    "model": model,
    "model_name": "iris-random-forest",
    "model_version": "1.0.0",
    "accuracy": acc,
}
joblib.dump(artifact, "models/iris_model.joblib")
print(artifact["model_name"], artifact["model_version"], acc)


## FastAPI: `app_fastapi.py`

Save the code below as `app_fastapi.py`, then run:

```bash
pip install fastapi uvicorn scikit-learn joblib pydantic
uvicorn app_fastapi:app --reload --port 8000
```

The API documentation will be available under `/docs`.


In [ ]:
# app_fastapi.py
from typing import List
import joblib
from fastapi import FastAPI
from pydantic import BaseModel, Field

artifact = joblib.load("models/iris_model.joblib")
model = artifact["model"]

class PredictRequest(BaseModel):
    features: List[float] = Field(..., min_length=4, max_length=4)

class PredictResponse(BaseModel):
    prediction: int
    model_name: str
    model_version: str

app = FastAPI(title="Iris ML API")

@app.get("/health")
def health():
    return {"status": "ok", "model_version": artifact["model_version"]}

@app.post("/predict", response_model=PredictResponse)
def predict(request: PredictRequest):
    pred = int(model.predict([request.features])[0])
    return PredictResponse(
        prediction=pred,
        model_name=artifact["model_name"],
        model_version=artifact["model_version"],
    )


Test FastAPI:

```bash
curl -X POST http://127.0.0.1:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"features": [5.1, 3.5, 1.4, 0.2]}'
```


## Flask: `app_flask.py`

Flask is useful for showing a minimal API. FastAPI is more convenient for validation and documentation.


In [ ]:
# app_flask.py
import joblib
from flask import Flask, request, jsonify

artifact = joblib.load("models/iris_model.joblib")
model = artifact["model"]
app = Flask(__name__)

@app.get("/health")
def health():
    return jsonify({"status": "ok", "model_version": artifact["model_version"]})

@app.post("/predict")
def predict():
    payload = request.get_json(force=True)
    features = payload["features"]
    pred = int(model.predict([features])[0])
    return jsonify({
        "prediction": pred,
        "model_name": artifact["model_name"],
        "model_version": artifact["model_version"],
    })

if __name__ == "__main__":
    app.run(port=5000, debug=True)


## Discussion points

- The API should return not only the prediction, but also the model version.
- The `/health` endpoint helps monitor the service.
- Input validation is critical.
- In production, it is worth adding a Dockerfile, logging, request limits, latency monitoring and error handling.
